# 🔍 Hướng dẫn sử dụng `jdeprscan`

**jdeprscan** là công cụ đi kèm JDK (từ JDK 9+) dùng để quét các API **deprecated** (đã lỗi thời) trong file `.class` hoặc JAR. Rất hữu ích khi nâng cấp JDK để biết code nào đang dùng API sắp bị gỡ bỏ.

## Các cách gọi jdeprscan
| Lệnh | Mô tả |
|---|---|
| `jdeprscan --release 17 myapp.jar` | Quét JAR, kiểm tra deprecated API theo JDK 17 |
| `jdeprscan --release 17 --verbose myapp.jar` | Chi tiết hơn, liệt kê cả deprecated API không được dùng |
| `jdeprscan --list --release 17` | Liệt kê tất cả deprecated API trong JDK 17 |
| `jdeprscan --for-removal --release 17 myapp.jar` | Chỉ hiện API **sẽ bị gỡ hoàn toàn** (forRemoval=true) |

In [1]:
import subprocess
import shutil
import os
import json
import concurrent.futures
from pathlib import Path

# ── 1. Tìm jdeprscan trên hệ thống ──
def find_jdeprscan() -> str | None:
    """Tìm executable jdeprscan trong PATH, JAVA_HOME, hoặc thư mục JDK phổ biến."""
    # Cách 1: tìm trong PATH
    direct = shutil.which("jdeprscan")
    if direct:
        return direct

    # Cách 2: tìm cạnh java executable
    java_exec = shutil.which("java")
    if java_exec:
        bin_dir = Path(java_exec).resolve().parent
        for name in ("jdeprscan.exe", "jdeprscan"):
            candidate = bin_dir / name
            if candidate.exists():
                return str(candidate)

    # Cách 3: dùng JAVA_HOME
    java_home = os.getenv("JAVA_HOME")
    if java_home:
        for name in ("jdeprscan.exe", "jdeprscan"):
            candidate = Path(java_home) / "bin" / name
            if candidate.exists():
                return str(candidate)

    # Cách 4: tìm trong thư mục JDK phổ biến trên Windows
    if os.name == "nt":
        common_roots = [
            os.getenv("ProgramFiles"),
            os.getenv("ProgramFiles(x86)"),
        ]
        for root in common_roots:
            if not root:
                continue
            for jdk_dir in Path(root).glob("Java/jdk*"):
                for name in ("jdeprscan.exe", "jdeprscan"):
                    candidate = jdk_dir / "bin" / name
                    if candidate.exists():
                        return str(candidate)

    return None

jdeprscan_path = find_jdeprscan()
print(f"jdeprscan found: {jdeprscan_path or '❌ NOT FOUND'}")

jdeprscan found: C:\Program Files\Java\jdk-17\bin\jdeprscan.exe


In [2]:
# ── 2. Liệt kê tất cả deprecated API trong JDK 17 ──
if jdeprscan_path:
    result = subprocess.run(
        [jdeprscan_path, "--list", "--release", "17"],
        capture_output=True, text=True, timeout=30
    )
    lines = result.stdout.strip().splitlines()
    print(f"Tổng số deprecated API trong JDK 17: {len(lines)}")
    print("\n📌 10 deprecated API đầu tiên:")
    for line in lines[:10]:
        print(f"  {line}")
else:
    print("SKIP: jdeprscan không khả dụng")

Tổng số deprecated API trong JDK 17: 658

📌 10 deprecated API đầu tiên:
  @Deprecated(since="1.7") javax.xml.stream.XMLEventFactory javax.xml.stream.XMLEventFactory.newInstance(java.lang.String,java.lang.ClassLoader)
  @Deprecated(since="1.7") javax.xml.stream.XMLInputFactory javax.xml.stream.XMLInputFactory.newInstance(java.lang.String,java.lang.ClassLoader)
  @Deprecated(since="1.7") javax.xml.stream.XMLInputFactory javax.xml.stream.XMLOutputFactory.newInstance(java.lang.String,java.lang.ClassLoader)
  @Deprecated javax.crypto.interfaces.DHPrivateKey.serialVersionUID
  @Deprecated javax.crypto.interfaces.DHPublicKey.serialVersionUID
  @Deprecated javax.crypto.interfaces.PBEKey.serialVersionUID
  @Deprecated(since="9") java.awt.Rectangle javax.swing.plaf.TextUI.modelToView(javax.swing.text.JTextComponent,int)
  @Deprecated(since="9") java.awt.Rectangle javax.swing.plaf.TextUI.modelToView(javax.swing.text.JTextComponent,int,javax.swing.text.Position.Bias)
  @Deprecated(since="9") int j

In [3]:
# ── 3. Chuẩn bị file JAR mẫu để quét ──
# Biên dịch và đóng gói lớp DeprecatedSample thành JAR
if jdeprscan_path:
    bin_dir = Path(jdeprscan_path).parent
    javac = bin_dir / ("javac.exe" if os.name == "nt" else "javac")
    jar_tool = bin_dir / ("jar.exe" if os.name == "nt" else "jar")

    smoke_dir = Path("artifacts/jdeprscan_smoke")
    java_file = smoke_dir / "DeprecatedSample.java"
    class_dir = smoke_dir / "classes"
    class_dir.mkdir(exist_ok=True)
    jar_file = smoke_dir / "deprecated-sample.jar"

    # Biên dịch
    compile_result = subprocess.run(
        [str(javac), "-d", str(class_dir), str(java_file)],
        capture_output=True, text=True
    )
    if compile_result.returncode == 0:
        print("✅ Biên dịch thành công")
        # Đóng gói JAR
        jar_result = subprocess.run(
            [str(jar_tool), "cf", str(jar_file), "-C", str(class_dir), "."],
            capture_output=True, text=True
        )
        print(f"✅ Tạo JAR: {jar_file} (exists={jar_file.exists()})")
    else:
        print(f"❌ Lỗi biên dịch: {compile_result.stderr}")
else:
    print("SKIP: jdeprscan không khả dụng")

✅ Biên dịch thành công
✅ Tạo JAR: artifacts\jdeprscan_smoke\deprecated-sample.jar (exists=True)


In [4]:
# ── 4. Chạy jdeprscan trên JAR mẫu ──
if jdeprscan_path and jar_file.exists():
    # Quét cơ bản
    result = subprocess.run(
        [jdeprscan_path, "--release", "17", str(jar_file)],
        capture_output=True, text=True, timeout=30
    )
    print("=== jdeprscan --release 17 deprecated-sample.jar ===")
    print(result.stdout or "(no stdout)")
    if result.stderr:
        print("STDERR:", result.stderr)

    print("\n--- Phân tích kết quả ---")
    for line in result.stdout.strip().splitlines():
        if "deprecated" in line.lower():
            print(f"⚠️  {line.strip()}")

=== jdeprscan --release 17 deprecated-sample.jar ===
Jar file artifacts\jdeprscan_smoke\deprecated-sample.jar:
class DeprecatedSample uses deprecated method java/lang/Thread::stop()V 


--- Phân tích kết quả ---
⚠️  Jar file artifacts\jdeprscan_smoke\deprecated-sample.jar:
⚠️  class DeprecatedSample uses deprecated method java/lang/Thread::stop()V


In [5]:
# ── 5. Dùng wrapper run_jdeprscan_check từ project ──
# Project MYGRATE đã có sẵn wrapper Python trong src/tools/maven_upgrade_tools.py
import sys
sys.path.insert(0, str(Path(".").resolve()))

from src.tools.maven_upgrade_tools import run_jdeprscan_check

jar_path = "artifacts/jdeprscan_smoke/deprecated-sample.jar"
if Path(jar_path).exists():
    report = run_jdeprscan_check(jar_path, target_release="17")
    print(json.dumps(report, indent=2, ensure_ascii=False))
else:
    print("SKIP: JAR chưa được tạo")

{
  "status": "WARN",
  "executable": "C:\\Program Files\\Java\\jdk-17\\bin\\jdeprscan.exe",
  "exit_code": 0,
  "finding_count": 1,
  "findings": [
    {
      "line": "class DeprecatedSample uses deprecated method java/lang/Thread::stop()V"
    }
  ],
  "output": "Jar file artifacts/jdeprscan_smoke/deprecated-sample.jar:\nclass DeprecatedSample uses deprecated method java/lang/Thread::stop()V"
}


## 📝 Tóm tắt

| Bước | Lệnh / Hàm | Mục đích |
|------|-----------|----------|
| Tìm tool | `find_jdeprscan()` | Tìm `jdeprscan` trên hệ thống |
| Liệt kê API | `jdeprscan --list --release 17` | Xem tất cả deprecated API trong JDK 17 |
| Quét JAR | `jdeprscan --release 17 app.jar` | Phát hiện code dùng deprecated API |
| Quét chi tiết | `jdeprscan --release 17 --verbose app.jar` | Bao gồm cả deprecated API không được dùng |
| Chỉ forRemoval | `jdeprscan --for-removal --release 17 app.jar` | Chỉ hiện API **sẽ bị gỡ hoàn toàn** |
| Wrapper Python | `run_jdeprscan_check(jar, release)` | Dùng từ code Python, trả dict có status/findings |

> ⚠️ `jdeprscan` chỉ phân tích **bytecode** (không cần source code). Nó phát hiện API deprecated ở **thời điểm biên dịch**, không phải runtime.

# 🧪 Dùng jdeprscan khoanh vùng code cần upgrade — Case study: Cantor
## Workflow: compile JDK 8 -> scan jdeprscan JDK 17 -> khoanh vùng -> quyết định upgrade
## Tại sao 2 JDK?
- Cantor pom.xml phụ thuộc jdk.tools (tools.jar) -> chỉ JDK 8 mới có
- jdeprscan cần bytecode đã compile -> phải compile trước
- jdeprscan --release 17 check bytecode theo deprecated list của JDK 17

In [6]:
CANTOR_PATH = Path(r"D:\capstone_project\MYGRATE---Capstone-Project\freshbrew_data\cantor")

# ── Dùng tool từ src/tools/jdeprscan_pipeline.py ──
# Tool này đã được tổng quát hóa: không hardcode Hadoop, không hardcode ecosystem,
# chỉ chèn tools.jar khi Maven dependency list có jdk.tools
import sys
sys.path.insert(0, str(Path(".").resolve()))
from src.tools.jdeprscan_pipeline import run_jdeprscan_pipeline

# ── Chạy full pipeline B0-B3 ──
result = run_jdeprscan_pipeline(
    project_path=str(CANTOR_PATH),
    target_release="17",
    jdk8_home=str(Path(os.getenv("LOCALAPPDATA", "")) / "Java" / "jdk-8"),
    logger=lambda msg: print(msg),
)

# ── In kết quả tổng hợp ──
print(f"\n{'═'*60}")
print(f"  KẾT QUẢ: {result['status']}")
print(f"  Project: {result['project_path']}")
print(f"  Target:  JDK {result['target_release']}")
print(f"  JDK 8:   {result['jdk8_home']}")
print(f"  JDK 17:  {result['jdk17_home']}")
if result['errors']:
    print(f"  Errors:  {result['errors']}")
print(f"{'═'*60}")

# ── B0: Maven ──
b0 = result['steps']['b0_maven']
print(f"\n── B0: Maven ──")
print(f"  Status: {b0['status']}, JARs: {b0['jar_count']}, needs_tools_jar: {b0['needs_tools_jar']}")

# ── B1: Compile ──
b1 = result['steps']['b1_compile']
print(f"\n── B1: Compile ──")
print(f"  Status: {b1['status']}, Classes: {b1['class_count']}, JAR: {b1['jar_path']}")

# ── B2: Project scan ──
b2 = result['steps']['b2_project_scan']
print(f"\n── B2: Project scan ──")
print(f"  Status: {b2['status']}, forRemoval: {b2['for_removal_count']}, deprecated: {b2['deprecated_count']}")
if b2.get('for_removal'):
    print("  !! forRemoval (sẽ gỡ hẳn):")
    for l in b2['for_removal']:
        print(f"     {l}")
if b2.get('deprecated'):
    print("  ! deprecated (chưa gỡ):")
    for l in b2['deprecated']:
        print(f"     {l}")

# ── B3: Dependencies scan ──
b3 = result['steps']['b3_dep_scan']
print(f"\n── B3: Dependencies scan ──")
print(f"  Status: {b3['status']}, Total: {b3['total_jars']}, Problem: {len(b3['problem_jars'])}, Clean: {b3['clean_jars']}, Timeout: {len(b3['timeout_jars'])}")

if b3['problem_jars']:
    ranked = sorted(b3['problem_jars'], key=lambda s: (s['for_removal'], s['total']), reverse=True)
    print("\n  -- Top nghiêm trọng (forRemoval nhiều nhất) --")
    for s in ranked[:15]:
        tag = "!!" if s['for_removal'] >= 10 else "!" if s['for_removal'] > 0 else " "
        print(f"  {tag} {s['jar']:<40} forRemoval={s['for_removal']:>3}  total={s['total']:>3}")
    if len(ranked) > 15:
        print(f"  ... +{len(ranked)-15} JARs khác")

    print("\n  -- Phân nhóm theo ecosystem (tự động theo prefix JAR) --")
    for eco_name, eco_data in b3['ecosystems'].items():
        fr = eco_data['for_removal_total']
        tot = eco_data['deprecated_total']
        n = len(eco_data['jars'])
        tag = "!!" if fr >= 10 else "!" if fr > 0 else " "
        print(f"  {tag} {eco_name:<18} {n:>2} JARs, forRemoval={fr:>3}, total={tot:>3}")

if b3['timeout_jars']:
    print("\n  -- Timeout (JAR quá lớn, jdeprscan không quét xong 180s) --")
    for s in b3['timeout_jars']:
        print(f"  ⏳ {s['jar']}")

# ── Tổng hợp 3 lớp ──
print(f"\n{'─'*55}")
print("TỔNG HỢP — 3 lớp cần xử lý khi upgrade JDK 17")
print(f"{'─'*55}")

s = result['summary']

# Lớp 1: project code
print(f"\n  1. Project code:")
if s['project_code']['clean']:
    print(f"     -> Sạch, không cần sửa gì")
else:
    if s['project_code']['for_removal_count'] > 0:
        print(f"     !! {s['project_code']['for_removal_count']} API forRemoval -> phải sửa, không thì crash runtime")
    if s['project_code']['deprecated_count'] > 0:
        print(f"     !  {s['project_code']['deprecated_count']} API deprecated -> nên sửa, vẫn chạy nhưng cảnh báo")

# Lớp 2: dependencies
print(f"\n  2. Dependencies:")
if s['dependencies']['problem_count'] > 0:
    for issue in s['dependencies']['top_issues']:
        print(f"     !! {issue['jar']}: {issue['for_removal']} forRemoval / {issue['total']} total")
    print(f"     -> Nâng version hoặc thay lib cho {s['dependencies']['problem_count']} JARs có deprecated API")
else:
    print(f"     -> Không có dependency nào dùng deprecated API")

# Lớp 3: pom.xml — phân tích từ dữ liệu, không hardcode
print(f"\n  3. pom.xml:")
if s['pom_xml']['critical_count'] > 0:
    print(f"     !! {s['pom_xml']['critical_count']} dependency có forRemoval -> phải nâng version hoặc thay lib:")
    for dep in s['pom_xml']['critical_deps']:
        print(f"        {dep['jar']}: {dep['for_removal']} forRemoval / {dep['total']} total")
else:
    print(f"     -> Không có dependency nào có forRemoval")

print(f"\n  Workflow: compile JDK 8 -> scan jdeprscan JDK 17")
print(f"  JDK 8:  {result['jdk8_home']}")
print(f"  JDK 17: {result['jdk17_home']}")

[jdeprscan] Khám phá JDK...
[jdeprscan] JDK 8:  C:\Users\tngtr\AppData\Local\Java\jdk-8
[jdeprscan] JDK 17: C:\Program Files\Java\jdk-17
[jdeprscan] jdeprscan: C:\Program Files\Java\jdk-17\bin\jdeprscan.exe
[jdeprscan] Maven: C:\Users\tngtr\AppData\Local\Apache\apache-maven-3.9.16\bin\mvn
[jdeprscan] B0: Maven resolve dependencies...
[jdeprscan] B0: cache hit — 54 JARs, pom.xml không đổi
[jdeprscan] B0: jdk.tools dependency CÓ -> sẽ chèn tools.jar
[jdeprscan] B1: Compile project...
[jdeprscan] B1: tools.jar added to classpath
[jdeprscan] B1: compiling 3 source files...
[jdeprscan] B1: OK — 2 class files
[jdeprscan] B2: jdeprscan project JAR...
[jdeprscan] B2: scanning cantor.jar...
[jdeprscan] B2: 0 forRemoval, 0 deprecated
[jdeprscan] B3: jdeprscan dependencies...
[jdeprscan] B3: scanning 54 JARs với 4 luồng...
[jdeprscan] B3: 35 problem, 19 clean, 0 timeout

════════════════════════════════════════════════════════════
  KẾT QUẢ: OK
  Project: D:\capstone_project\MYGRATE---Capstone-Pr